# CNN para classificação multiclasse de imagens
## Visualização dos filtros e dos *feature maps* — MNIST

Ficha resolvida com:
- download/preparação do MNIST;
- normalização;
- `batch_size = 32`;
- holdout treino/validação;
- 4 modelos CNN;
- treino com `CrossEntropyLoss` + `SGD`;
- avaliação com previsões e matriz de confusão;
- visualização de filtros, pesos, max pooling e *feature maps*.


## 0. Instalação de dependências e imports


In [ ]:
# Executar esta célula primeiro.
# Instala automaticamente apenas os pacotes que estiverem em falta.

import sys
import subprocess
import importlib.util


def install_if_missing(import_name, package_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name or import_name])

# Em Colab normalmente torch/torchvision já existem.
install_if_missing("torch", "torch")
install_if_missing("torchvision", "torchvision")
install_if_missing("torchinfo", "torchinfo")
install_if_missing("livelossplot", "livelossplot")
install_if_missing("sklearn", "scikit-learn")
install_if_missing("seaborn", "seaborn")
install_if_missing("pandas", "pandas")
install_if_missing("numpy", "numpy")
install_if_missing("matplotlib", "matplotlib")
install_if_missing("gdown", "gdown")


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import Module, Sequential, Conv2d, ReLU, MaxPool2d, Linear, Softmax, BatchNorm2d, Dropout
from torch.utils.data import DataLoader, random_split, Subset

from torchvision import datasets, transforms
from torchvision.transforms import Compose, ToTensor, Normalize

from torchinfo import summary
from livelossplot import PlotLosses

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("torch:", torch.__version__)
print("cuda disponível:", torch.cuda.is_available())


In [ ]:
# Constantes gerais
PATH = "./"
DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 32
RANDOM_SEED = 42
VALIDATION_SPLIT = 0.20

# Mudar para False se quiseres obrigar a treinar mesmo quando já existem .pth guardados.
LOAD_MODEL_IF_EXISTS = True

# Valores pedidos no enunciado.
EPOCHS_CNN1 = 15
EPOCHS_CNN2 = 15
EPOCHS_CNN3 = 15
EPOCHS_CNN4 = 75
LEARNING_RATE = 0.001

# Para testes rápidos antes da submissão, podes mudar para True.
# Para entregar, deixa False.
FAST_DEV_RUN = False
FAST_TRAIN_BATCHES = 80
FAST_VAL_BATCHES = 30

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
def get_default_device():
    """Devolve GPU se existir; caso contrário devolve CPU."""
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


def to_device(data, device):
    """Move tensors/listas/tuplos para o device escolhido."""
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)


class DeviceDataLoader:
    """Wrapper para mover batches automaticamente para GPU/CPU."""
    def __init__(self, dataloader, device):
        self.dataloader = dataloader
        self.device = device

    def __iter__(self):
        for batch in self.dataloader:
            yield to_device(batch, self.device)

    def __len__(self):
        return len(self.dataloader)


device = get_default_device()
print(device)


## 1. Preparar os dados


In [ ]:
# Transformações pedidas: conversão para tensor + normalização.
# Valores standard do MNIST: mean=0.1307, std=0.3081.

train_transform = Compose([
    ToTensor(),
    Normalize((0.1307,), (0.3081,))
])

test_transform = Compose([
    ToTensor(),
    Normalize((0.1307,), (0.3081,))
])


In [ ]:
def prepare_data_loaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE, validation_split=VALIDATION_SPLIT):
    """Faz download do MNIST e prepara train/validation/test loaders com holdout."""

    train_dataset_for_train = datasets.MNIST(
        root=data_dir,
        train=True,
        download=True,
        transform=train_transform
    )

    train_dataset_for_val = datasets.MNIST(
        root=data_dir,
        train=True,
        download=True,
        transform=test_transform
    )

    test_dataset = datasets.MNIST(
        root=data_dir,
        train=False,
        download=True,
        transform=test_transform
    )

    total_size = len(train_dataset_for_train)
    val_size = int(total_size * validation_split)
    train_size = total_size - val_size

    generator = torch.Generator().manual_seed(RANDOM_SEED)
    indices = torch.randperm(total_size, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]

    train_dataset = Subset(train_dataset_for_train, train_indices)
    val_dataset = Subset(train_dataset_for_val, val_indices)

    train_dl_raw = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
    val_dl_raw = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
    test_dl_raw = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

    train_dl = DeviceDataLoader(train_dl_raw, device)
    val_dl = DeviceDataLoader(val_dl_raw, device)
    test_dl = DeviceDataLoader(test_dl_raw, device)

    return train_dl, val_dl, test_dl, train_dl_raw, val_dl_raw, test_dl_raw


train_dl, val_dl, test_dl, train_dl_all, val_dl_all, test_dl_all = prepare_data_loaders()

print("Nº batches treino:", len(train_dl))
print("Nº batches validação:", len(val_dl))
print("Nº batches teste:", len(test_dl))


## 1.1 Visualizar os dados


In [ ]:
def output_label(label, mapping="extenso"):
    labels = {
        0: "zero",
        1: "um",
        2: "dois",
        3: "três",
        4: "quatro",
        5: "cinco",
        6: "seis",
        7: "sete",
        8: "oito",
        9: "nove"
    }
    label = int(label)
    if mapping == "label":
        return label
    return labels[label]


list_classes = [output_label(i) for i in range(10)]
list_classes


In [ ]:
def denormalize_mnist(img):
    return img * 0.3081 + 0.1307


def visualize_mnist_images(dl, n_images=32):
    images, labels = next(iter(dl))
    images = images.cpu()
    labels = labels.cpu()

    n_images = min(n_images, len(images))
    n_cols = 8
    n_rows = int(np.ceil(n_images / n_cols))

    plt.figure(figsize=(14, 2 * n_rows))
    for i in range(n_images):
        plt.subplot(n_rows, n_cols, i + 1)
        img = denormalize_mnist(images[i, 0]).clamp(0, 1)
        plt.imshow(img, cmap="gray")
        plt.title(output_label(labels[i]))
        plt.axis("off")
    plt.tight_layout()
    plt.show()


visualize_mnist_images(train_dl_all)


## 2. Definir os modelos


In [ ]:
class CNNModel_1(Module):
    def __init__(self):
        super().__init__()
        self.layer1 = Sequential(
            Conv2d(1, 32, kernel_size=3, stride=1),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = Sequential(
            Conv2d(32, 32, kernel_size=3, stride=1),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc1 = Linear(32 * 5 * 5, 100)
        self.act1 = ReLU()
        self.fc2 = Linear(100, 10)
        self.act2 = Softmax(dim=1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        return x


model = CNNModel_1().to(device)
print(summary(model, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
class CNNModel_2(Module):
    def __init__(self):
        super().__init__()
        self.layer1 = Sequential(
            Conv2d(1, 32, kernel_size=3, stride=1),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = Sequential(
            Conv2d(32, 32, kernel_size=3, stride=1),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc1 = Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x


model = CNNModel_2().to(device)
print(summary(model, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
class CNNModel_3(Module):
    def __init__(self):
        super().__init__()
        self.layer1 = Sequential(
            Conv2d(1, 32, kernel_size=3, stride=1),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = Sequential(
            Conv2d(32, 32, kernel_size=3, stride=1),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc1 = Linear(32 * 5 * 5, 128)
        self.dropout = Dropout(p=0.5)
        self.fc2 = Linear(128, 64)
        self.fc3 = Linear(64, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


model = CNNModel_3().to(device)
print(summary(model, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
class CNNModel_4(Module):
    def __init__(self):
        super().__init__()
        self.layer1 = Sequential(
            Conv2d(1, 32, kernel_size=3, stride=1),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(p=0.25)
        )
        self.layer2 = Sequential(
            Conv2d(32, 32, kernel_size=3, stride=1),
            BatchNorm2d(32),
            ReLU(),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(p=0.25)
        )
        self.fc1 = Linear(32 * 5 * 5, 128)
        self.fc2 = Linear(128, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = CNNModel_4().to(device)
print(summary(model, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


## 3. Treinar os modelos


In [ ]:
def accuracy_from_outputs(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds), device=labels.device)


def batch_loss_accuracy(model, batch, criterion):
    images, labels = batch
    outputs = model(images)
    loss = criterion(outputs, labels)
    acc = accuracy_from_outputs(outputs, labels)
    return loss, acc


@torch.no_grad()
def evaluate_epoch(model, val_dl, criterion, max_batches=None):
    model.eval()
    losses = []
    accuracies = []

    for batch_idx, batch in enumerate(val_dl):
        if max_batches is not None and batch_idx >= max_batches:
            break
        loss, acc = batch_loss_accuracy(model, batch, criterion)
        losses.append(loss.detach())
        accuracies.append(acc.detach())

    return {
        "val_loss": torch.stack(losses).mean().item(),
        "val_acc": torch.stack(accuracies).mean().item()
    }


def train_one_epoch(model, train_dl, criterion, optimizer, max_batches=None):
    model.train()
    losses = []
    accuracies = []

    for batch_idx, batch in enumerate(train_dl):
        if max_batches is not None and batch_idx >= max_batches:
            break

        loss, acc = batch_loss_accuracy(model, batch, criterion)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.detach())
        accuracies.append(acc.detach())

    return {
        "train_loss": torch.stack(losses).mean().item(),
        "train_acc": torch.stack(accuracies).mean().item()
    }


def train_model(model_path, train_dl, val_dl, model, criterion, optimizer, epochs):
    history = []
    plotlosses = PlotLosses()

    max_train_batches = FAST_TRAIN_BATCHES if FAST_DEV_RUN else None
    max_val_batches = FAST_VAL_BATCHES if FAST_DEV_RUN else None

    for epoch in range(epochs):
        train_metrics = train_one_epoch(model, train_dl, criterion, optimizer, max_batches=max_train_batches)
        val_metrics = evaluate_epoch(model, val_dl, criterion, max_batches=max_val_batches)

        metrics = {**train_metrics, **val_metrics}
        history.append(metrics)

        plotlosses.update({
            "loss": metrics["train_loss"],
            "val_loss": metrics["val_loss"],
            "accuracy": metrics["train_acc"],
            "val_accuracy": metrics["val_acc"]
        })
        plotlosses.send()

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"train_loss={metrics['train_loss']:.4f} | train_acc={metrics['train_acc']:.4f} | "
            f"val_loss={metrics['val_loss']:.4f} | val_acc={metrics['val_acc']:.4f}"
        )

    torch.save(model.state_dict(), model_path)
    print("Modelo guardado em:", model_path)
    return pd.DataFrame(history)


def build_or_load_model(model_class, model_name, epochs):
    model_path = MODEL_DIR / f"{model_name}.pth"
    model = model_class().to(device)

    if LOAD_MODEL_IF_EXISTS and model_path.exists():
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        print(f"{model_name} carregado de {model_path}")
        return model, pd.DataFrame()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)
    history = train_model(model_path, train_dl, val_dl, model, criterion, optimizer, epochs)
    return model, history


In [ ]:
######### CNNModel_1 ################
model_1, history_1 = build_or_load_model(CNNModel_1, "CNNModel_1", EPOCHS_CNN1)
print(summary(model_1, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
######### CNNModel_2 ################
model_2, history_2 = build_or_load_model(CNNModel_2, "CNNModel_2", EPOCHS_CNN2)
print(summary(model_2, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
######### CNNModel_3 ################
model_3, history_3 = build_or_load_model(CNNModel_3, "CNNModel_3", EPOCHS_CNN3)
print(summary(model_3, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
######### CNNModel_4 ################
model_4, history_4 = build_or_load_model(CNNModel_4, "CNNModel_4", EPOCHS_CNN4)
print(summary(model_4, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


In [ ]:
trained_models = {
    "CNNModel_1": model_1,
    "CNNModel_2": model_2,
    "CNNModel_3": model_3,
    "CNNModel_4": model_4,
}

histories = {
    "CNNModel_1": history_1,
    "CNNModel_2": history_2,
    "CNNModel_3": history_3,
    "CNNModel_4": history_4,
}


## 4. Avaliar os modelos


In [ ]:
@torch.no_grad()
def evaluate_model(test_dl, model):
    model.eval()
    actual_values = []
    predictions = []

    for images, labels in test_dl:
        outputs = model(images)
        _, preds = torch.max(outputs, dim=1)
        actual_values.extend(labels.detach().cpu().numpy().tolist())
        predictions.extend(preds.detach().cpu().numpy().tolist())

    acc = accuracy_score(actual_values, predictions)
    cm = confusion_matrix(actual_values, predictions)
    return actual_values, predictions, cm, acc


def display_predictions(actual_values, predictions, n=30):
    df = pd.DataFrame({
        "real": actual_values[:n],
        "real_extenso": [output_label(x) for x in actual_values[:n]],
        "previsto": predictions[:n],
        "previsto_extenso": [output_label(x) for x in predictions[:n]],
        "correto": [a == p for a, p in zip(actual_values[:n], predictions[:n])]
    })
    display(df)


def display_confusion_matrix(cm, list_classes):
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=list_classes, yticklabels=list_classes)
    plt.xlabel("Previsto")
    plt.ylabel("Real")
    plt.title("Matriz de confusão")
    plt.tight_layout()
    plt.show()


In [ ]:
evaluation_results = {}

for model_name, model in trained_models.items():
    print("=" * 80)
    print(model_name)

    actual_values, predictions, cm, acc = evaluate_model(test_dl, model)
    evaluation_results[model_name] = {
        "actual_values": actual_values,
        "predictions": predictions,
        "cm": cm,
        "accuracy": acc
    }

    print(f"Accuracy teste: {acc:.4f}")
    print(classification_report(actual_values, predictions, target_names=list_classes))
    display_predictions(actual_values, predictions, n=30)
    display_confusion_matrix(cm, list_classes)


## 5. Usar os modelos


In [ ]:
def img_show(img, legenda):
    img = img.detach().cpu()
    plt.figure(figsize=(3, 3))
    plt.axis("off")
    plt.title(legenda)
    plt.grid(False)
    plt.imshow(denormalize_mnist(img[0, 0]).clamp(0, 1), cmap=plt.get_cmap("gray"))
    plt.show()


def make_prediction(model, img):
    model.eval()
    img = img.reshape(1, 1, 28, 28)
    print(img.shape)
    print(img.dtype)
    img = img.to(device)

    with torch.no_grad():
        prediction = model(img).detach().cpu().numpy()[0].argmax()

    legenda = f"predict: {prediction} ({output_label(prediction)})"
    img_show(img, legenda)
    return prediction


imagens, labels = next(iter(test_dl))
make_prediction(model_1, imagens[3])


In [ ]:
def show_batch_images(model, dataloader, num_images=32):
    model.eval()
    images, labels = next(iter(dataloader))

    with torch.no_grad():
        outputs = model(images)
        _, preds = torch.max(outputs, dim=1)

    images_cpu = images.detach().cpu()
    labels_cpu = labels.detach().cpu()
    preds_cpu = preds.detach().cpu()

    num_images = min(num_images, len(images_cpu))
    n_cols = 8
    n_rows = int(np.ceil(num_images / n_cols))

    plt.figure(figsize=(14, 2 * n_rows))
    for i in range(num_images):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(denormalize_mnist(images_cpu[i, 0]).clamp(0, 1), cmap="gray")
        ok = labels_cpu[i].item() == preds_cpu[i].item()
        title = f"R:{labels_cpu[i].item()} P:{preds_cpu[i].item()}"
        if not ok:
            title += " ✗"
        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    return images, preds


for model_name, model in trained_models.items():
    print(model_name)
    images, pred = show_batch_images(model, test_dl, num_images=32)


In [ ]:
# Imprimir os modelos usados
for model_name, model in trained_models.items():
    print("=" * 80)
    print(model_name)
    print(model)


## 5.1 Visualizar parâmetros/filtros dos modelos


In [ ]:
def plot_filters_single_channel_big(t):
    nrows = t.shape[0] * t.shape[2]
    ncols = t.shape[1] * t.shape[3]
    npimg = np.array(t.numpy(), np.float32)
    npimg = npimg.transpose((0, 2, 1, 3))
    npimg = npimg.ravel().reshape(nrows, ncols)
    npimg = npimg.T
    fig, ax = plt.subplots(figsize=(max(ncols / 10, 4), max(nrows / 200, 2)))
    sns.heatmap(npimg, xticklabels=False, yticklabels=False, cmap="gray", ax=ax, cbar=False)
    plt.show()


def plot_filters_single_channel(t):
    nplots = t.shape[0] * t.shape[1]
    ncols = 12
    nrows = int(np.ceil(nplots / ncols))

    count = 0
    fig = plt.figure(figsize=(ncols, max(nrows, 1)))

    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            count += 1
            ax1 = fig.add_subplot(nrows, ncols, count)
            npimg = np.array(t[i, j].numpy(), np.float32)
            std = np.std(npimg)
            if std != 0:
                npimg = (npimg - np.mean(npimg)) / std
            npimg = np.minimum(1, np.maximum(0, (npimg + 0.5)))
            ax1.imshow(npimg, cmap="gray")
            ax1.set_title(str(i) + "," + str(j), fontsize=7)
            ax1.axis("off")
            ax1.set_xticklabels([])
            ax1.set_yticklabels([])

    plt.tight_layout()
    plt.show()


def plot_filters_multi_channel(t):
    num_kernels = t.shape[0]
    num_cols = 12
    num_rows = int(np.ceil(num_kernels / num_cols))

    fig = plt.figure(figsize=(num_cols, max(num_rows, 1)))
    for i in range(num_kernels):
        ax1 = fig.add_subplot(num_rows, num_cols, i + 1)
        npimg = np.array(t[i].numpy(), np.float32)
        std = np.std(npimg)
        if std != 0:
            npimg = (npimg - np.mean(npimg)) / std
        npimg = np.minimum(1, np.maximum(0, (npimg + 0.5)))
        npimg = npimg.transpose((1, 2, 0))
        ax1.imshow(npimg)
        ax1.axis("off")
        ax1.set_title(str(i), fontsize=7)
        ax1.set_xticklabels([])
        ax1.set_yticklabels([])

    plt.tight_layout()
    plt.show()


def plot_weights(layer, single_channel=True, collated=False):
    if isinstance(layer, nn.Conv2d):
        weight_tensor = layer.weight.data.cpu()
        if single_channel:
            if collated:
                plot_filters_single_channel_big(weight_tensor)
            else:
                plot_filters_single_channel(weight_tensor)
        else:
            if weight_tensor.shape[1] == 3:
                plot_filters_multi_channel(weight_tensor)
            else:
                print("Só é possível visualizar pesos com 3 canais quando single_channel=False.")
    else:
        print("Só é possível visualizar camadas convolucionais.")


def visualize_model_parameters(model):
    for name, param in model.named_parameters():
        print(f"{name:30s} | shape={tuple(param.shape)} | requires_grad={param.requires_grad}")


In [ ]:
for model_name, model in trained_models.items():
    print("=" * 80)
    print(model_name)
    visualize_model_parameters(model)

    print("Filtros da primeira camada convolucional:")
    plot_weights(model.layer1[0], single_channel=True)

    print("Filtros da segunda camada convolucional:")
    plot_weights(model.layer2[0], single_channel=True)


## 5.2 Aplicar max pooling


In [ ]:
def visualize_max_pooling(dataloader):
    images, labels = next(iter(dataloader))
    img = images[0:1].to(device)

    pool = nn.MaxPool2d(kernel_size=2, stride=2)
    pooled = pool(img)

    original = denormalize_mnist(img.detach().cpu()[0, 0]).clamp(0, 1)
    pooled_img = pooled.detach().cpu()[0, 0]

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(original, cmap="gray")
    plt.title(f"Original 28x28 — label {labels[0].item()}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(pooled_img, cmap="gray")
    plt.title("Depois de MaxPool2d 14x14")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_max_pooling(test_dl)


## 5.3 Percorrer camadas convolucionais/pooling e visualizar feature maps


In [ ]:
def get_conv_layers(model):
    conv_layers = []
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            conv_layers.append(layer)
    return conv_layers


def get_conv_pool_layers(model):
    conv_pool_layers = []
    for layer in model.modules():
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            conv_pool_layers.append(layer)
    return conv_pool_layers


for model_name, model in trained_models.items():
    conv_layers = get_conv_layers(model)
    conv_pool_layers = get_conv_pool_layers(model)
    print("=" * 80)
    print(model_name)
    print("Camadas convolucionais:", conv_layers)
    print("Nº camadas convolucionais:", len(conv_layers))
    print("Camadas conv/pool:", conv_pool_layers)
    print("Nº camadas conv/pool:", len(conv_pool_layers))


In [ ]:
def percorrer_conv_layers(model, images):
    """Executa o modelo e guarda os outputs das camadas Conv2d/MaxPool2d."""
    model.eval()
    outputs = []
    layer_names = []
    handles = []

    def hook_fn(name):
        def hook(module, inputs, output):
            outputs.append(output.detach().cpu())
            layer_names.append(name)
        return hook

    for name, layer in model.named_modules():
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            handles.append(layer.register_forward_hook(hook_fn(name)))

    with torch.no_grad():
        _ = model(images.to(device))

    for handle in handles:
        handle.remove()

    return outputs, layer_names


# Exemplo com o CNNModel_1
images, labels = next(iter(test_dl))
outputs, layer_names = percorrer_conv_layers(model_1, images)

print(f"Obtiveram-se {len(outputs)} tensores com o shape:")
for i, output in enumerate(outputs):
    print(f"Layer {i} ({layer_names[i]}) - {output.shape}")


In [ ]:
def visualize_featureMaps_partial(outputs, layer_names=None, num_imagem=0, max_filters=18):
    for num_layer in range(len(outputs)):
        layer_viz = outputs[num_layer][num_imagem, :, :, :]
        layer_viz = layer_viz.data

        title = f"Layer {num_layer + 1}"
        if layer_names is not None:
            title += f" — {layer_names[num_layer]}"
        print(title)

        n_filters = min(max_filters, layer_viz.shape[0])
        n_cols = 6
        n_rows = int(np.ceil(n_filters / n_cols))

        plt.figure(figsize=(14, 2.4 * n_rows))
        for i in range(n_filters):
            plt.subplot(n_rows, n_cols, i + 1)
            plt.imshow(layer_viz[i].cpu(), cmap="gray")
            plt.title(f"FM {i}")
            plt.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close()


visualize_featureMaps_partial(outputs, layer_names, num_imagem=6)


In [ ]:
# Visualização dos feature maps para todos os modelos.
# Pode gerar muitas imagens, mas cobre o que é pedido no enunciado.

images, labels = next(iter(test_dl))

for model_name, model in trained_models.items():
    print("=" * 80)
    print(model_name)
    outputs, layer_names = percorrer_conv_layers(model, images)

    print(f"Obtiveram-se {len(outputs)} tensores com o shape:")
    for i, output in enumerate(outputs):
        print(f"Layer {i} ({layer_names[i]}) - {output.shape}")

    visualize_featureMaps_partial(outputs, layer_names, num_imagem=6, max_filters=18)


## 6. Conclusão

O notebook executa todas as tarefas da ficha:

- T1/T2: download do MNIST e dependências;
- T3: `batch_size = 32`;
- T4: transformações, normalização e data loaders com holdout;
- T5: labels por extenso e batch de treino visualizado;
- T6: definição dos 4 modelos CNN;
- T7: treino dos 4 modelos com os epochs, learning rate, loss e optimizer pedidos;
- T8: avaliação, previsões e matriz de confusão;
- T9: uso dos modelos, visualização de parâmetros, filtros, max pooling e feature maps;
- T10: notebook pronto a submeter depois de correr todas as células.
